Imports:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import sys
sys.path.insert(0, "..")

from src.data.loader import download_prices, format_prices, get_ticker_data
from src.models.ridge import RidgeForecaster
from src.models.attention import AttentionForecaster
from src.evaluation.walk_forward import walk_forward, run_evaluation

from src.models.modules import LagFeatureTokeniser, LatentQueryAttention
from src.models.lag_feature_forecaster import LagFeatureForecaster
from src.evaluation.purged_kfold import PurgedWalkForwardSplit
from src.evaluation.purged_kfold import PurgedTrainValSplit

sns.set_style("whitegrid")
SEED = 40304451
np.random.seed(SEED)

%matplotlib inline
plt.rcParams["figure.dpi"] = 150

# Lag-Feature Attention Model Fine Tuning

---

## 1. Lag–Feature Tokenisation

### Motivation

Standard attention mechanisms operate on a sequence of vectors. Our input is a 2-D lookback window $X_{t-L+1:t} \in \mathbb{R}^{L \times F}$, where $L$ is the number of lags and $F$ is the number of features. We need to convert this grid into a flat sequence of $d$-dimensional vectors that an attention layer can consume.

### Token Construction

For each lag $\ell \in \{0, \ldots, L-1\}$ and feature $f \in \{1, \ldots, F\}$, we construct a token:

$$
u_{t,\ell,f} = \phi\!\left(x_{t-\ell,f}\right) + p_\ell + e_f \in \mathbb{R}^d
$$

where:

- $\phi : \mathbb{R} \to \mathbb{R}^d$ is a learned linear map (`scalar_embed = nn.Linear(1, d)`) that lifts each scalar observation into the embedding space.
- $p_\ell \in \mathbb{R}^d$ is a **learned lag embedding** (`lag_embed = nn.Embedding(L, d)`), that gives the model a sense of temporal position.
- $e_f \in \mathbb{R}^d$ is a **learned feature embedding** (`feature_embed = nn.Embedding(F, d)`), giving the model a sense of which variable it is looking at.

All $L \times F$ tokens are stacked to form:

$$
U_t \in \mathbb{R}^{(L \cdot F) \times d}
$$

### Code Reference: `src/models/modules.py :: LagFeatureTokeniser`

The three terms map directly to the `forward` method:

```python
phi_out = self.scalar_embed(x.unsqueeze(-1)) # phi(x_{t-l,f})
lag_vecs = self.lag_embed(self.lag_indices) # p_l
feat_vecs = self.feature_embed(self.feat_indices) # e_f
tokens = phi_out + lag_vecs + feat_vecs # u_{t,l,f}
```

The final `.reshape(batch, L * F, d)` collapses the 2-D `(L, F)` grid into a flat token sequence, preserving row-major ordering: `(l=0,f=0), (l=0,f=1), ..., (l=L-1,f=F-1)`.


### What the Embeddings Learn

It is worth pausing on why we add $p_\ell$ and $e_f$ rather than simply passing $\phi(x_{t-\ell,f})$ directly.

Without position information, the attention mechanism is **permutation-invariant**: swapping the order of tokens produces the same output. Since lag ordering carries temporal meaning (lag 0 is the most recent observation), we must inject this structure explicitly. The lag embedding $p_\ell$ is the analogue of positional encoding in Transformer models — it tells the model *when* a token was observed. The feature embedding $e_f$ tells it *what* was observed. Together they break the permutation symmetry in both dimensions.

In [2]:
BATCH = 8
L = 10
F = 5
D = 32
M = 4

print(f"Demo data dimensions: batch={BATCH}, lags={L}, features={F}")
print(f"Model hyperparameters: embed_dim={D}, num_queries={M}")

X_synthetic = torch.randn(BATCH, L, F)
print(f"Input window X shape: {tuple(X_synthetic.shape)}  (batch, L, F)")

tokeniser = LagFeatureTokeniser(num_lags=L, num_features=F, embed_dim=D)

U = tokeniser(X_synthetic)
print(f"Token matrix U shape: {tuple(U.shape)}  (batch, L*F, d)")

expected_tokens = L * F
assert U.shape == (BATCH, expected_tokens, D), "Unexpected token shape"
print(f"Shape check passed: {BATCH} x {expected_tokens} x {D}")

print(f"\nToken stats across batch:")
print(f" mean = {U.mean().item():.4f}")
print(f" std = {U.std().item():.4f}")
print(f" min = {U.min().item():.4f}")
print(f" max = {U.max().item():.4f}")

Demo data dimensions: batch=8, lags=10, features=5
Model hyperparameters: embed_dim=32, num_queries=4
Input window X shape: (8, 10, 5)  (batch, L, F)
Token matrix U shape: (8, 50, 32)  (batch, L*F, d)
Shape check passed: 8 x 50 x 32

Token stats across batch:
 mean = -0.0244
 std = 1.5996
 min = -5.2948
 max = 5.6729


---

## 2. Latent-Query QKV Attention Bottleneck

### Motivation

We now have $U_t \in \mathbb{R}^{(L \cdot F) \times d}$ — a sequence of $L \cdot F$ tokens. Standard self-attention would produce another sequence of the same length, which is not what we want: we need a fixed-size representation to feed into a forecasting head. The latent-query mechanism solves this by using $m$ **learned query vectors** rather than queries derived from the input. These $m$ queries act as a learned compression: they attend over all $L \cdot F$ tokens and pool the result into $m$ summary vectors.

### The Attention Computation

Given $Q^{\text{latent}} \in \mathbb{R}^{m \times d}$ (learned parameters), compute:

$$
K_t = U_t W_K, \quad V_t = U_t W_V, \quad Q = Q^{\text{latent}} W_Q
$$

where $W_K, W_V \in \mathbb{R}^{d \times d_k}$ and $W_Q \in \mathbb{R}^{d \times d_k}$ are learned projections without bias (standard in attention layers).

The attention weights and context vectors are then:

$$
A_t = \text{softmax}\!\left(\frac{Q K_t^\top}{\sqrt{d_k}}\right) \in \mathbb{R}^{m \times (L \cdot F)}
$$

$$
Z_t = A_t V_t \in \mathbb{R}^{m \times d_v}
$$

The $\sqrt{d_k}$ scaling prevents the dot products from growing large as $d_k$ increases, which would push the softmax into regions of near-zero gradient.

### Code Reference — `src/models/modules.py :: LatentQueryAttention`

```python
K = self.W_K(tokens)                                    # (batch, L*F, d_k)
V = self.W_V(tokens)                                    # (batch, L*F, d_v)
Q = self.W_Q(self.Q_latent)                             # (m, d_k)

scores = torch.matmul(Q, K.transpose(-2, -1)) / scale  # (batch, m, L*F)
A      = F.softmax(scores, dim=-1)                     # (batch, m, L*F)
Z      = torch.matmul(A, V)                            # (batch, m, d_v)
```

Note that `Q` has shape `(m, d_k)` with no batch dimension. PyTorch broadcasts this across the batch dimension of `K` automatically, giving `scores` shape `(batch, m, L*F)`. This is correct and efficient — the latent queries are **shared across all samples in the batch**, as they are global learned parameters.

In [3]:
attention = LatentQueryAttention(embed_dim=D, num_queries=M)

Z, A = attention(U)

print(f"Context matrix Z shape: {tuple(Z.shape)}  (batch, m, d_v)")
print(f"Attention matrix A shape: {tuple(A.shape)}  (batch, m, L*F)")

assert Z.shape == (BATCH, M, D)
assert A.shape == (BATCH, M, L * F)
print("Shape checks passed")

row_sums = A[0].sum(dim=-1)
print(f"\nAttention row sums (should be 1.0 for each query):")
for query_idx, row_sum in enumerate(row_sums):
    print(f"  query {query_idx}: row_sum = {row_sum.item():.6f}")

Context matrix Z shape: (8, 4, 32)  (batch, m, d_v)
Attention matrix A shape: (8, 4, 50)  (batch, m, L*F)
Shape checks passed

Attention row sums (should be 1.0 for each query):
  query 0: row_sum = 1.000000
  query 1: row_sum = 1.000000
  query 2: row_sum = 1.000000
  query 3: row_sum = 1.000000


### Interpretable Outputs: Feature and Lag Weights

The attention matrix $A_t \in \mathbb{R}^{m \times (L \cdot F)}$ is the central interpretability object. The token dimension indexes the $(\ell, f)$ grid in row-major order, so we can reshape it to $\mathbb{R}^{m \times L \times F}$ and marginalise to obtain two interpretable weight series:

$$
w_{t,f} = \sum_{k=1}^{m} \sum_{\ell=0}^{L-1} A_t[k, \ell, f] \qquad \text{(feature importance at time } t\text{)}
$$

$$
w_{t,\ell} = \sum_{k=1}^{m} \sum_{f=1}^{F} A_t[k, \ell, f] \qquad \text{(lag importance at time } t\text{)}
$$

These are time-varying: at each test window $t$ the model produces a different $A_t$, so $w_{t,f}$ and $w_{t,\ell}$ evolve across the backtest period. Plotting their trajectories reveals which features and lags the model relies on in different market regimes.

In [4]:
A_grid = A.reshape(BATCH, M, L, F)
print(f"A reshaped to grid: {tuple(A_grid.shape)}  (batch, m, L, F)")

feature_weights = A_grid.sum(dim=(1, 2))
lag_weights     = A_grid.sum(dim=(1, 3))

print(f"Feature weights shape: {tuple(feature_weights.shape)}  (batch, F)")
print(f"Lag weights shape:     {tuple(lag_weights.shape)}  (batch, L)")

print("\nFeature weights for first sample in batch:")
for f_idx, w in enumerate(feature_weights[0]):
    print(f"feature {f_idx}: {w.item():.4f}")

print("\nLag weights for first sample in batch:")
for l_idx, w in enumerate(lag_weights[0]):
    print(f"lag {l_idx}: {w.item():.4f}")

A reshaped to grid: (8, 4, 10, 5)  (batch, m, L, F)
Feature weights shape: (8, 5)  (batch, F)
Lag weights shape:     (8, 10)  (batch, L)

Feature weights for first sample in batch:
feature 0: 0.7952
feature 1: 0.8777
feature 2: 0.8185
feature 3: 0.7891
feature 4: 0.7195

Lag weights for first sample in batch:
lag 0: 0.4705
lag 1: 0.3475
lag 2: 0.5707
lag 3: 0.4128
lag 4: 0.4481
lag 5: 0.3578
lag 6: 0.3636
lag 7: 0.2835
lag 8: 0.3788
lag 9: 0.3666


---

## `LagFeatureForecaster`

### The Full Forward Pass

The `LagFeatureForecaster` wires the tokeniser and attention module together into a complete forward pass, then applies a forecasting head $g(\cdot)$:

$$
\hat{y}_{t+h} = g\!\left(\text{Flatten}(Z_t)\right)
$$

where $\text{Flatten}(Z_t)$ reshapes $Z_t \in \mathbb{R}^{m \times d_v}$ to a vector of length $m \cdot d_v$, and $g$ is either a linear layer or a small MLP depending on `head_config`.

### Code Reference — `src/models/lag_feature_forecaster.py :: LagFeatureForecaster.forward`

```python
tokens = self.tokeniser(x)       # (batch, L*F, d)
Z, A   = self.attention(tokens)  # (batch, m, d_v), (batch, m, L*F)
z_flat = Z.reshape(batch, -1)    # (batch, m * d_v)
y_hat  = self.head(z_flat)       # (batch, 1)
```

The head input dimension is therefore $m \times d_v$. With the default `d_v = d = 32$ and $m = 4$, the head receives a 128-dimensional vector.

In [5]:
forecaster = LagFeatureForecaster(
    num_lags=L,
    num_features=F,
    embed_dim=D,
    num_queries=M,
    head_config=None
)

print(f"\nRunning forward pass with X shape: {tuple(X_synthetic.shape)}")
y_hat, A_out = forecaster(X_synthetic)

print(f"y_hat shape: {tuple(y_hat.shape)}  (batch, 1)")
print(f"A_out shape: {tuple(A_out.shape)}  (batch, m, L*F)")

assert y_hat.shape == (BATCH, 1)
assert A_out.shape == (BATCH, M, L * F)

print("\nPrediction stats:")
# Expect Noise:
print(f"mean = {y_hat.mean().item():.4f}")
print(f"std = {y_hat.std().item():.4f}")

Building head | input_dim=128
  num_lags=10, num_features=5
  embed_dim=32, num_queries=4
  head_config=None
  head_input_dim=128
  total trainable params: 3873

Running forward pass with X shape: (8, 10, 5)
y_hat shape: (8, 1)  (batch, 1)
A_out shape: (8, 4, 50)  (batch, m, L*F)

Prediction stats:
mean = -0.5302
std = 0.0182


## Shared Config

All tickers, dates, and evaluation parameters are declared once here.
Both models use identical inputs to ensure a fair comparison.

In [ ]:
TICKERS = [
    "XLK",   # Technology
    "XLP",   # Consumer Staples
    "XLV",   # Health Care
    "XLF",   # Financials
    "XLE",   # Energy
    "XLI",   # Industrials
    "XLY",   # Consumer Discretionary
    "XLU",   # Utilities
    "XLB",   # Materials
    "XLRE",  # Real Estate
    "XLC",   # Communication Services
]

DATE_DOWNLOAD_START = "2010-01-01"
DATE_DOWNLOAD_END   = "2024-12-31"
DATE_EVAL_START     = "2015-01-01"
DATE_EVAL_END       = "2019-12-31"

HORIZON = 5
N_LAGS = 20
STEP = 10

print(f"Tickers: {TICKERS}")
print(f"Download: {DATE_DOWNLOAD_START} to {DATE_DOWNLOAD_END}")
print(f"Eval: {DATE_EVAL_START} to {DATE_EVAL_END}")
print(f"Horizon: {HORIZON} days | Lags: {N_LAGS} | Step: {STEP} days")

# Our splitters facilitate purged k-fold cv with embargoed training observations (for every training split following a test split)
splitter = PurgedWalkForwardSplit(
    horizon=HORIZON,
    step=STEP,
    eval_start=DATE_EVAL_START,
    eval_end=DATE_EVAL_END,
    embargo=HORIZON,
    min_train_size=30,  
)

inner_splitter = PurgedTrainValSplit(val_fraction=0.2, horizon=HORIZON)

Tickers: ['XLK', 'XLP', 'XLV', 'XLF', 'XLE', 'XLI', 'XLY', 'XLU', 'XLB', 'XLRE', 'XLC']
Download: 2010-01-01 to 2024-12-31
Eval: 2015-01-01 to 2019-12-31
Horizon: 5 days | Lags: 20 | Step: 10 days


## Feature Configs

Feature configs control what `build_features` constructs.
Both models receive the same (X, y, dates) for a given feature config.

In [7]:
FEATURE_CONFIGS = {
    "lags_only": {
        "n_lags": N_LAGS,
        "parkinson_vol_windows": (),
    },
    "lags_and_pvol": {
        "n_lags": N_LAGS,
        "parkinson_vol_windows": (5, 10),
    },
}

print(f"Feature configs: {list(FEATURE_CONFIGS.keys())}")

Feature configs: ['lags_only', 'lags_and_pvol']


## Model Configs

We control our model hyperparameters through model configs.

In [8]:
# Baseline Attention Config
ATTENTION_CONFIG = {
    "embed_dim": 32,
    "num_queries": 4,
    "batch_size": 32,
    "max_epochs": 100,
    "patience": 10,
    "learning_rate": 1e-3,
    "grad_clip": 1.0,
}

# It's then easy to load our models like so
MODELS = {
    "attention": (AttentionForecaster, ATTENTION_CONFIG),
}

print(f"Models: {list(MODELS.keys())}")

Models: ['attention']


## Download Data

In [9]:
raw_prices = download_prices(TICKERS, start=DATE_DOWNLOAD_START, end=DATE_DOWNLOAD_END)
prices_df = format_prices(raw_prices)

print(f"DataFrame shape: {prices_df.shape}")
print(f"Tickers present: {prices_df['ticker'].unique().tolist()}")
prices_df.head()

XLK: 3773 rows downloaded
XLP: 3773 rows downloaded
XLV: 3773 rows downloaded
XLF: 3773 rows downloaded
XLE: 3773 rows downloaded
XLI: 3773 rows downloaded
XLY: 3773 rows downloaded
XLU: 3773 rows downloaded
XLB: 3773 rows downloaded
XLRE: 2322 rows downloaded
XLC: 1644 rows downloaded

Total rows before cleaning: 37923

Missing values per column after formatting:
Open      0
High      0
Low       0
Close     0
Volume    5
dtype: int64
DataFrame shape: (37923, 10)
Tickers present: ['XLB', 'XLC', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLRE', 'XLU', 'XLV', 'XLY']


,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits,Capital Gains,ticker
date,,,,,,,,,,
2010-01-04,16.790001,17.010000,16.725000,17.010000,12.016514,15135000.0,0.0,0.0,0.0,XLB
2010-01-05,17.040001,17.090000,16.905001,17.065001,12.055369,17678400.0,0.0,0.0,0.0,XLB
2010-01-06,17.070000,17.415001,17.040001,17.355000,12.260233,16184000.0,0.0,0.0,0.0,XLB
2010-01-07,17.270000,17.290001,17.080000,17.219999,12.164865,11514600.0,0.0,0.0,0.0,XLB
2010-01-08,17.180000,17.459999,17.139999,17.459999,12.334409,9284800.0,0.0,0.0,0.0,XLB


## Walk-Forward Experiment Loop

Iterates over tickers x feature configs x models.
Each combination is evaluated independently on the same walk-forward splits.

In [ ]:
results = {}

for ticker in TICKERS:
    print(f"\n{'='*60}")
    print(f"Ticker: {ticker}")
    print(f"{'='*60}")

    prices = get_ticker_data(prices_df, ticker)

    for feature_config_name, feature_config in FEATURE_CONFIGS.items():
        print(f"\nFeature config: {feature_config_name}")

        for model_name, (forecaster_cls, model_config) in MODELS.items():
            print(f"Model: {model_name}/n-------------------")

            combined_config = {**feature_config, **model_config}

            forecaster = forecaster_cls(combined_config)
            X, y, dates = forecaster.build_features(prices, horizon=HORIZON)

            fold_results = walk_forward(
                X=X,
                y=y,
                dates=dates,
                forecaster_cls=forecaster_cls,
                config=combined_config,
                splitter=splitter,
                inner_splitter=inner_splitter,
                verbose=True
            )

            metrics = run_evaluation(fold_results)
            metrics["ticker"] = ticker
            metrics["feature_config"] = feature_config_name
            metrics["model"] = model_name

            key = f"{ticker}_{feature_config_name}_{model_name}"
            results[key] = metrics

            print(f"  IC={metrics['ic']:.4f} | DirAcc={metrics['diracc']:.4f} | MAE={metrics['mae']:.6f}")


Ticker: XLK
XLK: 3773 rows, 2010-01-04 to 2024-12-30

Feature config: lags_only
Model: attention/n-------------------


AttentionForecaster | n_lags=20, pvol_windows=()
build_features | horizon=5
_make_features | shape=(3752, 20), columns=['ret_lag1', 'ret_lag2', 'ret_lag3', 'ret_lag4', 'ret_lag5', 'ret_lag6', 'ret_lag7', 'ret_lag8', 'ret_lag9', 'ret_lag10', 'ret_lag11', 'ret_lag12', 'ret_lag13', 'ret_lag14', 'ret_lag15', 'ret_lag16', 'ret_lag17', 'ret_lag18', 'ret_lag19', 'ret_lag20']
build_features | X=(3747, 20, 1), y=(3747,)
  date range: 2010-02-03 to 2024-12-20
  y mean=0.003234, std=0.027370
split | n_dates=3747, eval_positions=1258, eval_range=[2015-01-01, 2019-12-31]
fold 0 | test=[1237:1247] | (2015-01-02 to 2015-01-15) | train_cutoff=1227 | n_train=1227 | n_purged=10
fold 0 | n_train=1227, n_test=10, n_purged=10 | test_start=2015-01-02


AttentionForecaster | n_lags=20, pvol_windows=()
Building head | input_dim=128
  num_lags=20, num_features=1
  embed_dim=32, num_queries=4

## Results

In [ ]:
results_df = pd.DataFrame(results).T.reset_index(drop=True)
results_df = results_df[["ticker", "feature_config", "model", "ic", "diracc", "mae", "rmse"]]
results_df[["ic", "diracc", "mae", "rmse"]] = (
    results_df[["ic", "diracc", "mae", "rmse"]].astype(float).round(4)
)

results_df.to_csv("../results/notebook/01_experiment_results.csv", index=False)
print(results_df.to_string(index=False))

In [ ]:
summary = (
    results_df
    .groupby(["model", "feature_config"])[["ic", "diracc", "mae", "rmse"]]
    .mean()
    .round(4)
)
print("Mean metrics across all tickers:")
print(summary.to_string())